### Embedding

In [ ]:
# !pip --version

pip 25.0.1 from C:\pkh20260902\ex0922\.venv\Lib\site-packages\pip (python 3.12)



In [ ]:
# !pip install langchain_teddynote

In [3]:
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [ ]:
# !pip install langchain_openai

In [5]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [6]:
text = "This is a test sample for the embedding test."
query_result = embeddings.embed_query(text)
len(query_result)

1536

In [7]:
query_result[:5]

[0.03253173828125,
 0.006008148193359375,
 0.0271148681640625,
 -0.034423828125,
 0.0027217864990234375]

In [8]:
doc_result = embeddings.embed_documents(
    [text, text, text, text]
)

doc_result[0][:5]

[0.032501220703125,
 0.006008148193359375,
 0.0271453857421875,
 -0.034423828125,
 0.002704620361328125]

In [9]:
len(doc_result[0])

1536

In [12]:
embeddings_1024 = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

len(embeddings_1024.embed_documents([text])[0])

1024

In [ ]:
# !pip install scikit-learn

In [ ]:
# 유사도 확인
from sklearn.metrics.pairwise import cosine_similarity

sentence1 = "Hello, nice to meet you!"
sentence2 = "Hello, nice to meet you!"
sentence3 = "Hello, how are you doing?"
sentence4 = "Hi, today is sunny!"
sentence5 = "Do you like to eat bananas?"

sentences = [sentence1, sentence2, sentence3, sentence4, sentence5]
embedded_sentences = embeddings_1024.embed_documents(sentences)

In [16]:
def similarity(a, b):
    return cosine_similarity([a], [b])[0][0]

In [17]:
for i, sentence in enumerate(embedded_sentences):
    for j, other_sentence in enumerate(embedded_sentences):
        if i < j:
            print(
                f"[similarity{similarity(sentence, other_sentence):.4f}] {sentences[i]} \t <=====> \t {sentences[j]}"
            )

[similarity1.0000] Hello, nice to meet you! 	 <=====> 	 Hello, nice to meet you!
[similarity0.6635] Hello, nice to meet you! 	 <=====> 	 Hello, how are you doing?
[similarity0.4547] Hello, nice to meet you! 	 <=====> 	 Hi, today is sunny!
[similarity0.2248] Hello, nice to meet you! 	 <=====> 	 Do you like to eat bananas?
[similarity0.6635] Hello, nice to meet you! 	 <=====> 	 Hello, how are you doing?
[similarity0.4547] Hello, nice to meet you! 	 <=====> 	 Hi, today is sunny!
[similarity0.2248] Hello, nice to meet you! 	 <=====> 	 Do you like to eat bananas?
[similarity0.4254] Hello, how are you doing? 	 <=====> 	 Hi, today is sunny!
[similarity0.2813] Hello, how are you doing? 	 <=====> 	 Do you like to eat bananas?
[similarity0.1649] Hi, today is sunny! 	 <=====> 	 Do you like to eat bananas?


### Cache Backed Embeddings

In [ ]:
# !pip install langchain_community

In [ ]:
# !pip install faiss-cpu

In [20]:
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

embedding = OpenAIEmbeddings()

store = LocalFileStore("./cache/")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [22]:
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding,
    document_embedding_cache=store,
    namespace=embedding.model,
)

list(store.yield_keys())

[]

In [24]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader(r"langchain-kr\08-Embeddings\data\appendix-keywords.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)

In [30]:
%time db = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 62.5 ms
Wall time: 1.74 s


In [ ]:
# 처리 속도 up + 비용 절감
%time db2 = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 0 ns
Wall time: 9.95 ms


#### In Memory Byte Store (비영구적으로 임베딩을 저장)

In [ ]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import InMemoryByteStore

store = InMemoryByteStore() # 메모리 내 바이트 저장소 생성

# 캐시 지원 임베딩 생성
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding, store, namespace=embedding.model
)

### HuggingFace Embeddings

In [33]:
from dotenv import load_dotenv

load_dotenv()

True

In [35]:
from langchain_teddynote import logging
import os
import warnings

logging.langsmith("test0914")

warnings.filterwarnings("ignore")

os.environ["HF_HOME"] = "./cache/"

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [9]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. ",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

In [ ]:
# !pip install langchain_huggingface

In [38]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name,
    task="feature-extraction",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"]
)

In [39]:
%%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 375 ms
Wall time: 6.29 s


In [40]:
print("[HuggingFace Endpoint Embedding]")
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

[HuggingFace Endpoint Embedding]
Model: 		intfloat/multilingual-e5-large-instruct
Dimension: 	1024


In [41]:
embedded_query = hf_embeddings.embed_query("Langchain에 대해서 알려주세요.")
embedded_query

[0.004734391812235117,
 0.02326434664428234,
 -0.0028131757862865925,
 -0.021782316267490387,
 0.022929685190320015,
 -0.013653792440891266,
 -0.023130318149924278,
 0.03274926915764809,
 0.033536553382873535,
 -0.034004226326942444,
 0.021422840654850006,
 0.01692553423345089,
 -0.02530091069638729,
 -0.002398668322712183,
 -0.019956855103373528,
 -0.02397989109158516,
 -0.059594277292490005,
 0.008875398896634579,
 -0.02864757739007473,
 -0.014328647404909134,
 0.05715521425008774,
 0.006560272071510553,
 -0.013964306563138962,
 -0.01847483590245247,
 -0.0164940282702446,
 -0.0001644297008169815,
 -0.023537442088127136,
 -0.0407705157995224,
 0.004622231237590313,
 -0.034477945417165756,
 -0.0006146568921394646,
 0.009265496395528316,
 -0.023547319695353508,
 -0.046442531049251556,
 -0.01653599552810192,
 0.024837536737322807,
 0.048964451998472214,
 0.040364399552345276,
 -0.017567407339811325,
 0.052073486149311066,
 -0.020614419132471085,
 0.06236651539802551,
 0.02897622995078563

#### 임베딩된 질문과 문서 간의 유사도 계산하기

In [42]:
import numpy as np

# 질문(embedded_query): Langchain에 대해서 알려주세요.
np.array(embedded_query) @ np.array(embedded_documents).T

array([0.84504136, 0.84930372, 0.85086804, 0.8848799 , 0.76329368])

In [ ]:
# 내적 결과를 내림차순으로 정렬
sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]
sorted_idx

array([3, 2, 1, 0, 4])

In [44]:
# Query와 관련된 문서를 유사도 순서대로 출력
print("[Query] Langchain 에 대해서 알려주세요. \n =======================")
for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

[Query] Langchain 에 대해서 알려주세요. 
[0] LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.

[1] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. 

[2] LangChain simplifies the process of building applications with large language models

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



#### HuggingFace Embeddings

In [ ]:
# !pip install sentence-transformers

In [1]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

c:\pkh20260902\ex0922\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\pkh20260902\ex0922\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rjsgh\.cache\huggingface\hub\models--intfloat--multilingual-e5-large-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate de

In [4]:
%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 0 ns
Wall time: 0 ns


In [6]:
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

Model: 		intfloat/multilingual-e5-large-instruct
Dimension: 	1024


### BGE-M3 임베딩

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-m3"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

%time
embedded_documents = hf_embeddings.embed_documents(texts)

print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

c:\pkh20260902\ex0922\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rjsgh\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 43611.66it/s]


CPU times: total: 0 ns
Wall time: 0 ns
Model: 		BAAI/bge-m3
Dimension: 	1024


In [4]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. ",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

In [13]:
import numpy as np

embedded_query = hf_embeddings.embed_query("Langchain에 대해서 알려주세요.")
embedded_documents = hf_embeddings.embed_documents(texts)

np.array(embedded_query) @ np.array(embedded_documents).T

sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]

print("[Query] Langchain에 대해서 알려주세요.\n===============================")
for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

[Query] Langchain에 대해서 알려주세요.
[0] LangChain simplifies the process of building applications with large language models

[1] LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.

[2] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. 

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



#### Flag Embedding

In [ ]:
# !pip install -qU FlagEmbedding

In [15]:
from FlagEmbedding import BGEM3FlagModel

model_name = "BAAI/bge-m3"
bge_embeddings = BGEM3FlagModel(
    model_name, use_fp16=True
)

bge_embedded = bge_embeddings.encode(
    texts,
    batch_size=12,
    max_length=8192,
)["dense_vecs"]

bge_embedded.shape

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 520.71it/s]


(5, 1024)

In [16]:
from FlagEmbedding import BGEM3FlagModel

bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)

bge_encoded = bge_flagmodel.encode(texts, return_dense=True)

bge_encoded["dense_vecs"].shape

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 669.43it/s]


(5, 1024)

In [17]:
bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)
bge_encoded = bge_flagmodel.encode(texts,return_sparse=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 636.91it/s]


In [18]:
lexical_scores1 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded["lexical_weights"][0], bge_encoded["lexical_weights"][0]
)
lexical_scores2 = bge_flagmodel.compute_lexical_matching_score(
    bge_encoded["lexical_weights"][0], bge_encoded["lexical_weights"][1]
)

print(lexical_scores1)
print(lexical_scores2)

0.30156016
0


In [19]:
bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=True
)
bge_encoded = bge_flagmodel.encode(texts, return_colbert_vecs=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1542.61it/s]


In [20]:
colbert_scores1 = bge_flagmodel.colbert_score(
    bge_encoded["colbert_vecs"][0], bge_encoded["colbert_vecs"][0]
)

colbert_scores2 = bge_flagmodel.colbert_score(
    bge_encoded["colbert_vecs"][0], bge_encoded["colbert_vecs"][1]
)

print(colbert_scores1) # 같은 텍스트의 ColBERT 벡터를 비교하여 매칭 점수 확인
print(colbert_scores2) # 서로 다른 텍스트의 ColBERT 벡터 비교

tensor(1.0000)
tensor(0.3748)


### Upstage Embeddings

In [ ]:
# !pip install langchain_upstage

In [1]:
from langchain_upstage import UpstageEmbeddings
from langchain_teddynote import logging
from dotenv import load_dotenv

logging.langsmith("test0914")
load_dotenv()

# Query 전용 임베딩 모델
query_embeddings = UpstageEmbeddings(model="solar-embedding-1-large-query")

# 문서 전용 임베딩 모델
passage_embeddings = UpstageEmbeddings(model="solar-embedding-1-large-passage")

c:\pkh20260902\ex0922\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [7]:
embedded_query = query_embeddings.embed_query("LangChain에 대해서 상세히 알려주세요.")
len(embedded_query)

4096

In [ ]:
# 문서 임베딩
embedded_documents = passage_embeddings.embed_documents(texts)

In [8]:
import numpy as np

# 질문(embedded_query): LangChain에 대해서 알려주세요
similarity = np.array(embedded_query) @ np.array(embedded_documents).T

# 유사도 기준 내림차순 정렬
sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]

# 결과 출력
print("[Query] LangChain에 대해서 알려주세요.\n=========================================")
for i, idx in enumerate(sorted_idx):
    print(f"[{i}] similarity: {similarity[idx]:.3f} | {texts[idx]}")

[Query] LangChain에 대해서 알려주세요.
[0] similarity: 0.484 | LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.
[1] similarity: 0.464 | LangChain simplifies the process of building applications with large language models
[2] similarity: 0.434 | 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. 
[3] similarity: 0.187 | Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.
[4] similarity: 0.153 | 안녕, 만나서 반가워.
